# Proyecto Oráculo

Análisis y Diseño de Algoritmos

El trabajo es encontrar una **configuración de prompt** — un índice por ranura del catálogo, más la temperatura — que haga que el modelo cumpla restricciones verificables. Ustedes solo escriben en la **celda 4**. Todo lo demás ya está hecho.

El oráculo es una caja negra: le dan una config y un lote de instancias, y les devuelve la precisión y las trazas de lo que falló. Pueden consultar `evaluar` y `validar` cuantas veces quieran; lo ya generado no se vuelve a calcular.

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # partición de validación, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```

Cada instancia trae **hasta 5 restricciones** y se puntúa todo-o-nada: falla una, vale 0. Midan cada candidata sobre las mismas 40 instancias: con menos, los pasos cerca del piso se confunden con cero.

Cada ranura ofrece un consejo de prompting distinto, y algunos combinan mal con las restricciones que mide el verificador. Las trazas dicen qué restricción falló. Guarden cada `r.precision` y dibujen `curva(historial)`.

`espacio()` tiene 1024 configs con temperatura 0.0. `espacio(TEMPERATURAS)` agrega 0.3 y 0.7 y lo triplica. Recorrerlas todas no cabe en una T4.


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Baja `oraculo.py`, `ayudas.py` y `datos_visibles.json`, más las librerías que usan el modelo y los verificadores. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit (qwen8b, mistral7b). nltk/spacy/emoji/langdetect:
# los usa el verificador de open-instruct, no el notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.

`cargar_modelo` acepta un alias de la tabla o cualquier id público de Hugging Face (`org/nombre`). En `ayudas.py` hay más alias (`qwen8b`, `mistral7b`): caben en T4 en 4-bit, pero cada consulta tarda más.

| Alias | Checkpoint | Tamaño | Notas |
|---|---|---|---|
| `qwen17b` | `Qwen/Qwen3-1.7B` | 1.7B | el de por defecto; el más rápido |
| `ministral3b` | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.8B | fp16, ~7.7 GB de VRAM |
| `llama3b` | `unsloth/Llama-3.2-3B-Instruct` | 3.2B | fp16, ~6.4 GB de VRAM |

Son tres familias distintas (Qwen, Mistral, Llama): sirve para ver si su configuración generaliza o si solo le funciona a un modelo.

El caché guarda el nombre del modelo en la clave, así que cambiar de modelo **no** reusa respuestas del anterior: vuelve a gastar rollouts.

> Usen el repo `-BF16` de Ministral 3. El repo por defecto es FP8 y la T4 no lo soporta.


In [ ]:
from ayudas import cargar_modelo

# Alias → checkpoint. El nombre entra en la clave de caché: si cambian de
# modelo, las respuestas anteriores no se reusan.
#  "qwen17b"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
modelo = cargar_modelo("qwen17b")


## 3 · El oráculo

`dividir` parte `datos_visibles.json` en búsqueda (150) y validación (300). Busquen solo sobre `busqueda`. `validar` mide la config ya elegida sobre instancias que no se usaron al buscar: sirve para ver si generaliza a **instancias** nuevas, no a tipos de restricción nuevos.

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# Descomenten estas dos líneas si quieren que el caché sobreviva a una
# desconexión: el path de Drive reemplaza cache_oraculo.json local.
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
oraculo = Oraculo(modelo, busqueda, validacion)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/oraculo_cache.json")

# 1024 configs (temperatura 0.0). Para incluir 0.3 y 0.7: espacio(TEMPERATURAS)
CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Aquí escriben ustedes

Abajo hay una **búsqueda aleatoria** de ejemplo: saca configuraciones al azar del
espacio y se queda con la mejor. Está para que vean cómo se consulta el oráculo
y cómo se arma `curva(historial)`, no para imitarla.

Bórrenla y pongan su heurística. Dejen definidas `mejor` e `INSTANCIAS`: las celdas de
abajo (fallos, validación, entrega) las usan.


In [ ]:
# Ejemplo de búsqueda: aleatoria, con tope de evaluaciones.
# Bórrenlo y pongan su heurística.

import random

from ayudas import curva

random.seed(0)

# Tope pedagógico: el oráculo no limita consultas. 40 instancias fijas
# dejan ver pasos de 2.5%; con menos, cerca del piso todo parece cero.
MAX_EVALS = 10
INSTANCIAS = busqueda[:40]

mejor = None
historial = []
for i in range(MAX_EVALS):
    c = random.choice(CONFIGS)
    r = oraculo.evaluar(c, INSTANCIAS, semilla=1)
    historial.append(r.precision)

    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, c)

    print(f"eval {i + 1:3d}/{MAX_EVALS}   esta {r.precision:5.1%}   mejor {mejor[0]:5.1%}")

print("\nmejor configuración:", mejor[1])
curva(historial)

### Leer los fallos

Cada resultado trae sus trazas: mírenlas todas las veces que quieran. `violo` es la **primera** restricción que no pasó (el puntaje es todo-o-nada, no hace falta listar las demás). `salida` es el texto que produjo el modelo.

Si quieren ver el prompt **antes** de generar, `ver_prompt(config, instancia)` en `ayudas` arma el texto exacto que recibiría el modelo.


In [ ]:
# Reusa el caché: estas instancias ya se midieron al buscar.
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida

Las 300 instancias de validación no se usaron al buscar. Sirve para ver si la config aguanta instancias nuevas, no para elegir otra — si eligen con la validación, dejan de ser un conjunto de prueba.

Validarlas todas tarda; `n` escoge cuántas medir. La muestra es fija: las mismas n en cada llamada. Además imprime qué restricciones se cayeron más.


In [ ]:
# Escojan con cuántas instancias validar: más n = más confiable, pero más lento.
# n=30 es una muestra; n=None (o n=300) mide las 300.
r_val = oraculo.validar(mejor[1], n=30)

print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")


## 5 · La entrega

Un `entrega.json` con el grupo, la config ganadora y la semana. La config lleva un índice por ranura (`rol`, `estrategia`, `formato`, `estilo`, `cierre`) y `temperatura`. La nota no sale de este notebook: el profesor corre esa config sobre un test privado. Cambien `G07` y `semana` antes de descargar.


In [ ]:
from ayudas import entrega
from google.colab import files

# grupo: identificador del equipo. semana: número de semana del curso.
entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
